# vidcap demo

HDMI in → filter → timestamp → HDMI out on the PYNQ-Z2. Run the cells top to bottom with an HDMI source on the
input connector (1280x720 @ 60 Hz) and a monitor on the output. Without a source the board shows a colour-bar
test pattern after 5 seconds.

Copy this folder to `/home/xilinx/jupyter_notebooks/vidcap/` (the `make deploy` target does it).

**One notebook at a time.** Both notebooks load the bitstream, and only one process should drive the hardware. Before opening the other notebook, close this one (File → Close and Shut Down Notebook, or Kernel → Shut Down All Kernels). If the picture freezes or snapshots never finish, run `sudo python3 -m vidcap.diagnose` from this folder: it reads the hardware without changing it and says where the video stopped.

In [ ]:
import sys, time
sys.path.insert(0, '.')            # the 'vidcap' driver package sits next to this notebook
from vidcap import Vidcap

MODE = '720p'                    # '720p', '480p', '240p', '1080p30' or 'xga': each mode is its own bitstream
v = Vidcap.on_pynq(mode=MODE)     # downloads the bitstream and checks the board answers
v.start()                          # starts the DMA engines, shows live video, sets the clock from this board
v.status()

## What is the input doing?
`hdmi_good` means locked, the size of the loaded mode, frames arriving. `fallback_active` means the test pattern has taken over.

In [ ]:
v.input.status()

## Filters
The mode changes on a frame boundary, so there is no tearing.

In [ ]:
for mode in ['passthrough', 'invert', 'grayscale', 'threshold', 'sobel', 'blur', 'sharpen']:
    v.set_filter(mode, threshold=110)
    print(mode)
    time.sleep(2)
v.set_filter('passthrough')

## VCR noise
Adds grey grain from a free-running LFSR. `noise` is the strength: 0 = none, 255 = pixels move by up to ±127. The grain is different in every frame.

In [ ]:
v.set_filter('noise', noise=40);  time.sleep(4)     # light grain
v.set_filter('noise', noise=160); time.sleep(4)     # heavy snow
v.set_filter('passthrough')

## Temporal average
Averages the brightness of the last **N** frames (colour is kept), using a running sum in DDR, so the memory cost is the same for any N. The first frames after switching on or changing N average the frames available so far. N is limited by `history_frames` (16 by default; give `v.start(history_frames=60)` for more).

In [ ]:
for n in (1, 4, 16):
    v.set_temporal_average(n)
    time.sleep(3)
    print('N =', n, v.tavg.status())     # depth = frames in the average right now
v.set_temporal_average(None)      # off

## Colour grading
A 3D look-up table (17x17x17 colours, interpolated between the nodes) applied after the filter and before the timestamp. Pick a built-in look, set how strongly it applies, or load your own `.cube` file (the format DaVinci Resolve, Premiere and most grading tools export; any size, 1D or 3D). The interactive notebook has the same controls as widgets, including an upload button.

In [ ]:
for name, text in v.looks().items():
    print(f'{name:15s} {text}')

In [ ]:
for look in ('teal_orange', 'sepia', 'high_contrast', 'faded_film', 'bleach_bypass'):
    v.set_look(look)                      # loads the table, checks it, switches grading on (strength 100 %)
    print(look, v.status()['color_grade'])
    time.sleep(4)
v.set_look('teal_orange', strength=50)    # half-way between the original and the look
time.sleep(4)
v.set_look_strength(80)                   # change the strength without reloading the table
time.sleep(4)
v.set_look(None)                          # grading off

### Preview a look before you use it
`lut.apply_lut` is a bit-exact software model of the hardware, so you can see on screen what a look will do to a picture (here, the snapshot) before it goes to the video. If you have not taken a snapshot yet, the cell takes one.

In [ ]:
from vidcap import lut
from PIL import Image
import numpy as np
img = img if 'img' in globals() else v.snapshot()       # a picture to try the looks on
rgb = img[..., ::-1]                                   # the snapshot is B,G,R; the model takes R,G,B
row = [rgb] + [lut.apply_lut(rgb, lut.build_look(n)) for n in ('teal_orange', 'sepia', 'faded_film')]
strip = np.concatenate(row, axis=1)                    # original, then three looks side by side
Image.fromarray(strip).resize((strip.shape[1] // 3, strip.shape[0] // 3))

### Your own look
Upload a `.cube` file through Jupyter's file browser (the arrow button in the file list), then load it by path. The file is checked as it is read: a wrong row count or an unreadable line raises an error that says what is wrong. You can also write a built-in look to a `.cube` file, edit it in a grading tool, and load it back.

In [ ]:
# v.set_look('/home/xilinx/jupyter_notebooks/vidcap/my_look.cube', strength=100)

# save a built-in look as a .cube file you can open in other tools:
open('warm.cube', 'w').write(lut.cube_text(lut.build_look('warm')))

# or build one directly: a 17x17x17x3 uint8 array indexed [r][g][b] (here: swap the red and blue channels)
cube = lut.identity_cube()[..., ::-1].copy()
# v.set_look(cube)

## Test pattern
Colour bars with a moving box: if the box stops, the pipeline has stalled.

In [ ]:
v.use_test_pattern(True);  time.sleep(3)
v.use_test_pattern(False)

## Snapshot
Captures the next complete frame from the *selected input*, before filtering.

In [ ]:
img = v.snapshot()             # (height, width, 3) uint8 for the loaded mode, B,G,R order
from PIL import Image
Image.fromarray(img[..., ::-1]).resize((640, 360))

## Video memory
Show any image of the loaded mode's size instead of live video (it goes through the filters and gets the timestamp).

In [ ]:
import numpy as np
gradient = np.zeros(v.geo.shape, np.uint8)
gradient[..., 0] = np.linspace(0, 255, v.geo.width, dtype=np.uint8)           # blue ramp
gradient[..., 2] = np.linspace(0, 255, v.geo.height, dtype=np.uint8)[:, None]   # red ramp
v.show_image(gradient)
time.sleep(3)
v.show_image(img)                 # the captured snapshot, shown back
time.sleep(3)
v.show_live()

## Timestamp
The on-screen time is a hardware clock that was set from this board's clock at `start()`.

In [ ]:
v.overlay.configure(scale=3, margin=24, fg=(255, 255, 0), bg=(0, 0, 0))
print('hardware clock:', v.rtc.get(), ' ahead of the board clock by', round(v.rtc.drift_against(), 3), 's')

In [ ]:
# to correct drift: v.rtc.cycles_per_second(100_000_000 + adjustment), then v.sync_time()
v.overlay.enable(False)

## Diagnostics
VDMA error lists should be empty while video is flowing.

In [ ]:
v.status()